# 02 · Exploratory spatial analysis

1. Maps of lighting, population and crime rates in the urban study area
2. Lit vs unlit comparison (and Low / Medium / High lighting)
3. Global Moran's I: is crime spatially clustered?
4. LISA hot spots, and bivariate LISA of SLAI against street crime

Crime rates are per 1,000 residents and log-transformed (`log1p`) for the
spatial statistics, because they are highly right-skewed.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # lets the notebook import from src/

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from src import config

config.FIGURES.mkdir(parents=True, exist_ok=True)
config.TABLES.mkdir(parents=True, exist_ok=True)

from scipy import stats
from libpysal import weights
import esda

cells = gpd.read_file(config.PROCESSED_CELLS).set_index("cell_id")
urban = cells[cells.study_area.astype(bool)].copy()
for group in config.CRIME_GROUPS:
    urban[f"log_{group}_rate"] = np.log1p(urban[f"{group}_rate"])
print(f"{len(urban):,} study-area squares in {urban.settlement.nunique()} settlements")

## 1. Maps

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 11))
panels = [
    ("population", "Usual residents (Census 2021)", "viridis"),
    ("slai", "Street Light Availability Index", "cividis"),
    ("log_street_rate", "Street crime per 1,000 residents (log)", "magma"),
    ("log_comparison_rate", "Comparison crime per 1,000 residents (log)", "magma"),
]
for ax, (col, title, cmap) in zip(axes.flat, panels):
    cells.plot(ax=ax, color="#eeeeee", linewidth=0)          # all NI as background
    urban.plot(column=col, ax=ax, cmap=cmap, legend=True, linewidth=0,
               legend_kwds={"shrink": 0.6})
    ax.set_title(title)
    ax.set_axis_off()
plt.tight_layout()
plt.savefig(config.FIGURES / "fig01_maps.png", dpi=300, bbox_inches="tight")
plt.show()

## 2. Lit vs unlit areas

Almost every urban square has street lights, so the main test is Kruskal–Wallis across the
Low / Medium / High lighting classes. Mann–Whitney U (lit vs unlit) is only run if there are
enough unlit squares; across all of Northern Ireland, unlit squares are almost all rural. These are non-parametric because crime rates are skewed.
Differences here do not control for population or commercial activity;
that is what the models in notebook 03 are for.

In [ ]:
order = [c for c in ["Unlit", "Low", "Medium", "High"] if (urban.light_class == c).sum() >= 10]
summary = (urban.groupby("light_class")[[f"{g}_rate" for g in config.CRIME_GROUPS]]
           .agg(["count", "median", "mean"]).reindex(order))
summary.to_csv(config.TABLES / "rates_by_light_class.csv")
summary

In [ ]:
rows = []
for group in config.CRIME_GROUPS:
    rate = f"{group}_rate"
    lit, unlit = urban.loc[urban.lit == 1, rate], urban.loc[urban.lit == 0, rate]
    by_class = [urban.loc[urban.light_class == c, rate] for c in order if (urban.light_class == c).any()]
    mw = stats.mannwhitneyu(lit, unlit) if len(unlit) >= 10 else None
    kw = stats.kruskal(*by_class) if len(by_class) > 1 else None
    rows.append({
        "crime group": group,
        "Mann-Whitney U (lit vs unlit)": None if mw is None else mw.statistic,
        "p (lit vs unlit)": None if mw is None else mw.pvalue,
        "Kruskal-Wallis H (lighting classes)": None if kw is None else kw.statistic,
        "p (lighting classes)": None if kw is None else kw.pvalue,
    })
tests = pd.DataFrame(rows)
tests.to_csv(config.TABLES / "lit_vs_unlit_tests.csv", index=False)
tests

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, group in zip(axes, config.CRIME_GROUPS):
    data = [urban.loc[urban.light_class == c, f"log_{group}_rate"] for c in order]
    ax.boxplot(data, showfliers=False)
    ax.set_xticks(range(1, len(order) + 1), order)
    ax.set_title(f"{group.capitalize()} crime")
    ax.set_xlabel("Lighting class")
axes[0].set_ylabel("Crime per 1,000 residents (log)")
plt.tight_layout()
plt.savefig(config.FIGURES / "fig02_boxplots_light_class.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Global Moran's I

Spatial weights use the 8 nearest neighbours (row-standardised). K-nearest
neighbours are used instead of contiguity because urban cells form separate
clusters (towns), and contiguity weights would leave many cells with no neighbours.

In [ ]:
w = weights.KNN.from_dataframe(urban, k=8)
w.transform = "r"

moran_rows = []
for col in ["log_street_rate", "log_comparison_rate", "slai"]:
    mi = esda.Moran(urban[col].values, w, permutations=999)
    moran_rows.append({"variable": col, "Moran's I": mi.I, "p (permutation)": mi.p_sim})
moran_table = pd.DataFrame(moran_rows)
moran_table.to_csv(config.TABLES / "global_moran.csv", index=False)
moran_table

## 4. LISA and bivariate LISA

- **LISA** finds local clusters of high (HH) or low (LL) street crime, and outliers (HL, LH).
- **Bivariate LISA** asks whether a cell's lighting (SLAI) is related to street crime in neighbouring cells.

Only clusters with pseudo p < 0.05 are shown.

In [ ]:
LABELS = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}
COLOURS = {"High-High": "#d7191c", "Low-High": "#abd9e9", "Low-Low": "#2c7bb6",
           "High-Low": "#fdae61", "Not significant": "#dddddd"}

def lisa_labels(local, alpha=0.05):
    lab = pd.Series(local.q).map(LABELS)
    lab[local.p_sim >= alpha] = "Not significant"
    return lab.values

lisa = esda.Moran_Local(urban["log_street_rate"].values, w, permutations=999, seed=config.RANDOM_STATE)
urban["lisa_street"] = lisa_labels(lisa)

lisa_bv = esda.Moran_Local_BV(urban["slai"].values, urban["log_street_rate"].values, w,
                              permutations=999, seed=config.RANDOM_STATE)
urban["lisa_bv_slai_street"] = lisa_labels(lisa_bv)

bv = esda.Moran_BV(urban["slai"].values, urban["log_street_rate"].values, w, permutations=999)
print(f"Bivariate Moran's I (SLAI vs neighbouring street crime): {bv.I:.3f}, p = {bv.p_sim:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, col, title in [(axes[0], "lisa_street", "LISA: street crime"),
                       (axes[1], "lisa_bv_slai_street", "Bivariate LISA: SLAI vs street crime")]:
    cells.plot(ax=ax, color="#f5f5f5", linewidth=0)
    for label, colour in COLOURS.items():
        subset = urban[urban[col] == label]
        if len(subset):
            subset.plot(ax=ax, color=colour, linewidth=0, label=f"{label} ({len(subset)})")
    ax.set_title(title)
    ax.legend(loc="lower left", fontsize=8)
    ax.set_axis_off()
plt.tight_layout()
plt.savefig(config.FIGURES / "fig03_lisa.png", dpi=300, bbox_inches="tight")
plt.show()

urban[["lisa_street", "lisa_bv_slai_street"]].apply(pd.Series.value_counts)

For the bivariate map, read the first word as the cell's **lighting** and the
second as its neighbours' **street crime**: High-Low means well-lit cells
surrounded by low street crime.